In [46]:
!pip install Bio --quiet

### Step 1: 3D Structure Prediction (Swiss-Model) + Tertiary Identification (Foldseek)

In [47]:
import requests
import time
import os
import gzip

# --- 1A: 1D to 3D FOLDING ---
def fold_sequence(sequence, sm_token):
    print("--- STEP 1A: Predicting 3D Structure (Swiss-Model) ---")
    sm_submit_url = "https://swissmodel.expasy.org/automodel"
    headers = {"Authorization": f"Token {sm_token}", "Content-Type": "application/json"}
    payload = {"target_sequences": [sequence], "project_title": "Tertiary_Pipeline"}

    response = requests.post(sm_submit_url, headers=headers, json=payload)
    if response.status_code not in [200, 201, 202]:
        raise Exception(f"Swiss-Model submission failed: {response.text}")

    project_id = response.json().get('project_id')
    print(f"Job queued (Project ID: {project_id}). Polling for completion... (This usually takes 5-15 minutes)")

    status_url = f"https://swissmodel.expasy.org/project/{project_id}/models/summary/"

    while True:
        res = requests.get(status_url, headers=headers)

        if res.status_code == 200:
            data = res.json()
            if data.get("status") == "COMPLETED":
                models = data.get("models", [])
                if not models:
                    raise Exception("Swiss-Model finished but returned no models.")

                pdb_url = models[0].get('coordinates_url')
                pdb_response = requests.get(pdb_url, headers=headers)
                try:
                    pdb_data = gzip.decompress(pdb_response.content).decode('utf-8')
                except Exception:
                    pdb_data = pdb_response.text

                pdb_filename = f"predicted_{project_id}.pdb"

                with open(pdb_filename, "w") as f:
                    f.write(pdb_data)
                print(f"\n3D Structure predicted and saved as {pdb_filename}")
                return pdb_filename, pdb_data
            elif data.get("status") in ["FAILED", "ERROR"]:
                raise Exception("Swiss-Model job failed.")

        elif res.status_code == 202:
            print("Swiss-Model is still calculating... waiting 20 seconds.")
        else:
            print(f"Unexpected status code: {res.status_code}")

        time.sleep(20)

# --- 1B: 3D IDENTIFICATION ---
def identify_structure(pdb_filename, pdb_data):
    print("\n--- STEP 1B: Tertiary Identification (Foldseek) ---")
    fs_submit_url = "https://search.foldseek.com/api/ticket"
    files = {'q': (pdb_filename, pdb_data)}
    data = [('mode', '3diaa'), ('database[]', 'pdb100'), ('database[]', 'afdb50')]

    res = requests.post(fs_submit_url, files=files, data=data)
    if res.status_code != 200:
        raise Exception(f"Foldseek submission rejected! HTTP {res.status_code}: {res.text}")

    ticket_id = res.json().get('id')
    status_url = f"https://search.foldseek.com/api/ticket/{ticket_id}"

    while True:
        status_res = requests.get(status_url).json()
        status = status_res.get('status')

        if status == 'COMPLETE':
            break
        elif status in ['ERROR', 'FAILED']:
            print("\nFOLDSEEK ERROR: Foldseek rejected the PDB structure.")
            raise Exception("Foldseek rejected the Swiss-Model output.")

        time.sleep(2)

    result_url = f"https://search.foldseek.com/api/result/{ticket_id}/0"
    raw_results = requests.get(result_url).json()

    top_hits = {}

    if isinstance(raw_results, dict) and 'results' in raw_results:
        raw_results = raw_results['results']
    if not isinstance(raw_results, list):
        raw_results = [raw_results]

    for db_block in raw_results:
        if isinstance(db_block, dict) and 'alignments' in db_block:
            db_name = db_block.get('target', db_block.get('db', 'Unknown_DB'))
            alignments_array = db_block['alignments']

            if isinstance(alignments_array, list) and len(alignments_array) > 0:
                query_hits = alignments_array[0]

                if isinstance(query_hits, list) and len(query_hits) > 0:
                    top_hit = query_hits[0]
                    top_hits[db_name] = top_hit

                    target_name = top_hit.get('target', 'Unknown')
                    e_value = top_hit.get('eval', 'N/A')
                    print(f"Top match in {db_name}: {target_name} (E-value: {e_value})")

    if not top_hits:
        print("No significant structural matches found.")

    return top_hits

### Step 2: Physicochemical & Structural Profiling

In [48]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.PDB import PDBParser
import warnings

def analyze_properties(sequence, pdb_filename):
    print("\n--- STEP 2: Physicochemical & Structural Profile ---")

    analysis = ProteinAnalysis(sequence)
    mol_weight = analysis.molecular_weight()
    pI = analysis.isoelectric_point()
    gravy = analysis.gravy()
    instability = analysis.instability_index()

    print(f"1D Properties:")
    print(f"  - Molecular Weight: {mol_weight:.2f} Da")
    print(f"  - Isoelectric Point (pI): {pI:.2f}")
    print(f"  - GRAVY (Hydrophobicity): {gravy:.3f}")
    print(f"  - Instability Index: {instability:.2f}")

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        parser = PDBParser()
        structure = parser.get_structure("Predicted", pdb_filename)

    atoms = [atom for atom in structure.get_atoms()]
    coords = [atom.get_coord() for atom in atoms]

    if coords:
        x_span = max(c[0] for c in coords) - min(c[0] for c in coords)
        y_span = max(c[1] for c in coords) - min(c[1] for c in coords)
        z_span = max(c[2] for c in coords) - min(c[2] for c in coords)
    else:
        x_span = y_span = z_span = 0.0

    print(f"3D Properties:")
    print(f"  - Total Atoms in Model: {len(atoms)}")
    print(f"  - Spatial Dimensions: {x_span:.1f}Å x {y_span:.1f}Å x {z_span:.1f}Å")

    return {"pI": pI, "gravy": gravy, "instability": instability}

### Step 3: Actionable Wet-Lab Recommendations

In [49]:
def generate_wet_lab_report(props, foldseek_hits):
    print("\n--- STEP 3: Wet-Lab Recommendations ---")

    if props['gravy'] < 0:
        print("SOLUBILITY: Protein is hydrophilic. Standard cytosolic extraction buffers (e.g., Tris/PBS) without harsh detergents are recommended.")
    else:
        print("SOLUBILITY: Protein is highly hydrophobic. Consider adding mild detergents (e.g., Triton X-100 or CHAPS) during lysis.")

    if props['pI'] < 7.4:
        print("PURIFICATION: At physiological pH (7.4), this protein will be negatively charged. Use an Anion Exchange column.")
    else:
        print("PURIFICATION: At physiological pH (7.4), this protein will be positively charged. Use a Cation Exchange column.")

    if foldseek_hits:
        top_target = list(foldseek_hits.values())[0].get('target', 'Unknown')
        print(f"ASSAY DESIGN: Tertiary structure matches '{top_target}'. Review literature on this specific structural fold for known binding partners or enzyme activity assays.")
    else:
        print("ASSAY DESIGN: No exact tertiary structural matches found. Protein fold may be novel.")

In [50]:
TOKEN = "31f671a6084d2d6ec8b4959ea9e274f0b8a7c88b"

UNKNOWN_SEQ = "MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE"

try:
    pdb_file, pdb_data = fold_sequence(UNKNOWN_SEQ, TOKEN)
    hits = identify_structure(pdb_file, pdb_data)
    props = analyze_properties(UNKNOWN_SEQ, pdb_file)
    generate_wet_lab_report(props, hits)

    if os.path.exists(pdb_file):
        os.remove(pdb_file)

except Exception as e:
    print(f"\nPipeline Failed: {e}")

--- STEP 1A: Predicting 3D Structure (Swiss-Model) ---
Job queued (Project ID: bb35c0). Polling for completion... (This usually takes 5-15 minutes)

3D Structure predicted and saved as predicted_bb35c0.pdb

--- STEP 1B: Tertiary Identification (Foldseek) ---
Top match in afdb50: AF-A0A1S3FTQ6-F1-model_v6 Cytochrome c, somatic (E-value: 3.698e-20)
Top match in pdb100: 3zoo-assembly1.cif.gz_A Structure of the Y46F mutant of human cytochrome c (E-value: 9.9e-21)

--- STEP 2: Physicochemical & Structural Profile ---
1D Properties:
  - Molecular Weight: 11748.60 Da
  - Isoelectric Point (pI): 9.59
  - GRAVY (Hydrophobicity): -0.706
  - Instability Index: 11.38
3D Properties:
  - Total Atoms in Model: 824
  - Spatial Dimensions: 34.1Å x 33.1Å x 34.7Å

--- STEP 3: Wet-Lab Recommendations ---
SOLUBILITY: Protein is hydrophilic. Standard cytosolic extraction buffers (e.g., Tris/PBS) without harsh detergents are recommended.
PURIFICATION: At physiological pH (7.4), this protein will be positivel